# Flight Price Prediction - Final Clean Version

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics


## Load Data

In [ ]:
# ضع ملفاتك هنا (نفس اللي عندك بالمشروع)

df_1 = pd.read_csv("data/PAR_NYC.csv")
df_2 = pd.read_csv("data/PAR_SVO.csv")
df_3 = pd.read_csv("data/SVO_NYC.csv")
df_4 = pd.read_csv("data/SVO_RUH.csv")
df_5 = pd.read_csv("data/NYC_PAR.csv")
df_6 = pd.read_csv("data/NYC_SVO.csv")
df_7 = pd.read_csv("data/RUH_NYC.csv")
df_8 = pd.read_csv("data/RUH_PAR.csv")
df_9 = pd.read_csv("data/RUH_SVO.csv")
df_10 = pd.read_csv("data/SVO_PAR.csv")
df_11 = pd.read_csv("data/PAR_RUH.csv")
df_12 = pd.read_csv("data/NYC_RUH.csv")

dfs = [df_1,df_2,df_3,df_4,df_5,df_6,df_7,df_8,df_9,df_10,df_11,df_12]

df = pd.concat(dfs, ignore_index=True)


## Cleaning

In [ ]:
def clean_duration(duration):
    duration = list(duration)
    d = []
    for i in range(len(duration)):
        h = int(duration[i].split("h")[0])
        m = int(duration[i].split("m")[0].split()[-1])
        d.append(h*60 + m)
    return d

def clean_price(price):
    price = price.str.replace(',','',regex=True)
    price = price.str.replace('SAR','',regex=True)
    price = price.str.strip()
    price = pd.to_numeric(price) / 3.75
    return price

df["Duration"] = clean_duration(df["Duration"])
df["Price"] = clean_price(df["Price"])
df["Date"] = pd.to_datetime(df["Date"])


## Feature Engineering

In [ ]:
print("Before:", df.shape)

df["Duration_hours"] = df["Duration"] / 60
df["Day"] = df["Date"].dt.day
df["Month"] = df["Date"].dt.month
df["Weekday"] = df["Date"].dt.weekday

df["Log_Price"] = np.log(df["Price"])

# Remove Outliers
Q1 = df["Price"].quantile(0.25)
Q3 = df["Price"].quantile(0.75)
IQR = Q3 - Q1
low = Q1 - 1.5 * IQR
high = Q3 + 1.5 * IQR

df = df[(df["Price"] >= low) & (df["Price"] <= high)]

df["Price_per_hour"] = df["Price"] / df["Duration_hours"]

# Scaling
scaler = StandardScaler()
df[["Duration","Duration_hours","Price","Price_per_hour"]] = scaler.fit_transform(
    df[["Duration","Duration_hours","Price","Price_per_hour"]]
)

print("After:", df.shape)
df.head()


## Modeling

In [ ]:
X = df.select_dtypes(include=[np.number]).drop(["Price"], axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("MAE:", metrics.mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
print("R2:", metrics.r2_score(y_test, y_pred))


## Visualization

In [ ]:
plt.figure()
plt.scatter(y_test, y_pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()
